# BrainScan-v2 — Étape 2 : Détection des tumeurs cérébrales (YOLOv8)

## 2.1 — Exploration du dataset de détection (EDA)

**Objectifs du notebook**

1. Vérifier l'intégrité du dataset : structure YOLO (images/ + labels/), format des annotations.
2. Afficher un échantillon d'images avec leurs boîtes englobantes (bounding boxes) et classes.
3. Vérifier l'existence des fichiers .txt correspondants pour chaque image.
4. Organiser les données dans les dossiers images/labels selon train/valid/test.
5. Compter les images et annotations par ensemble.
6. Vérifier la validité du format YOLO et la cohérence des classes.
7. Créer le fichier data.yaml.

> **Note** : Ce notebook suppose que vous disposez d'un dataset au format YOLO
> (images + fichiers .txt avec annotations). Si vous n'avez pas de dataset de détection,
> vous pouvez utiliser le script `prepare.py` pour en créer un à partir de données brutes,
> ou télécharger un dataset public (ex: Brain Tumor Detection sur Kaggle/Roboflow).

In [ ]:
# Cellule 0 — Imports, configuration et chemins du projet
"""Cellules communes pour l'étape 2 (détection)."""

import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib

if matplotlib.get_backend().lower() == "agg":
    try:
        matplotlib.use("module://matplotlib_inline.backend_inline")
    except Exception:
        pass

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# --------------------------------------------------------------------------------------
# Configuration du projet
# --------------------------------------------------------------------------------------
def _detect_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").is_dir() and (candidate / "src").is_dir():
            return candidate
    if Path("/app/data").is_dir():
        return Path("/app")
    raise FileNotFoundError("Racine du projet introuvable")


PROJECT_ROOT = _detect_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DETECTION_DIR = DATA_DIR / "detection_raw"  # Données brutes détection (si disponibles)
PROCESSED_DETECTION_DIR = DATA_DIR / "processed" / "detection"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for _d in (ARTIFACTS_DIR, MODELS_DIR, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

SEED = 42

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RAW_DETECTION   : {RAW_DETECTION_DIR}")
print(f"PROCESSED_DETECT: {PROCESSED_DETECTION_DIR}")
print(f"ARTIFACTS_DIR   : {ARTIFACTS_DIR}")
print(f"MODELS_DIR      : {MODELS_DIR}")


### 2.2 — Structure attendue du dataset de détection

Pour YOLOv8, le dataset doit suivre cette structure :

```
detection_raw/
├── images/          # Toutes les images
│   ├── img_001.jpg
│   ├── img_002.jpg
│   └── ...
└── labels/          # Annotations YOLO (même nom de base)
    ├── img_001.txt
    ├── img_002.txt
    └── ...
```

Chaque fichier `.txt` contient une ligne par objet :
`class_id x_center y_center width height` (valeurs normalisées 0-1).

Après préparation (`prepare.py`), la structure devient :

```
processed/detection/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
├── test/
│   ├── images/
│   └── labels/
└── data.yaml
```

In [ ]:
# Cellule 2.1 — Vérification de l'existence des données brutes
if RAW_DETECTION_DIR.exists():
    print(f"✅ Dossier source trouvé : {RAW_DETECTION_DIR}")
    for sub in ["images", "labels"]:
        p = RAW_DETECTION_DIR / sub
        if p.exists():
            n = len(list(p.rglob("*")))
            print(f"  {sub}/ : {n} fichiers")
        else:
            print(f"  {sub}/ : MANQUANT")
else:
    print(f"⚠️  Dossier source non trouvé : {RAW_DETECTION_DIR}")
    print("   Vous devez fournir un dataset au format YOLO dans ce dossier.")
    print("   Structure attendue : detection_raw/{images,labels}/*.jpg + *.txt")


### 2.3 — Préparation du dataset (si données brutes disponibles)

Exécution du script de préparation pour organiser en train/valid/test et créer data.yaml.

In [ ]:
# Cellule 2.2 — Lancement de la préparation (si données brutes présentes)
import subprocess

if RAW_DETECTION_DIR.exists() and (RAW_DETECTION_DIR / "images").exists():
    cmd = [
        sys.executable, "-m", "brainscan.training.prepare",
        "--task", "detection",
        "--data-dir", str(RAW_DETECTION_DIR),
        "--output-dir", str(PROCESSED_DETECTION_DIR),
        "--split", "0.7", "0.15", "0.15",
    ]
    print(f"Exécution : {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
else:
    print("Données brutes non disponibles — préparation ignorée.")
    print("Placez vos données dans data/detection_raw/{images,labels}/ puis relancez.")


### 2.4 — Exploration du dataset préparé

Analyse de la structure train/valid/test, comptage, vérification annotations.

In [ ]:
# Cellule 2.3 — Chargement et analyse de data.yaml
yaml_path = PROCESSED_DETECTION_DIR / "data.yaml"

if yaml_path.exists():
    with open(yaml_path) as f:
        data_config = yaml.safe_load(f)
    print("data.yaml :")
    for k, v in data_config.items():
        print(f"  {k}: {v}")
    CLASS_NAMES = data_config.get("names", [])
    NC = data_config.get("nc", len(CLASS_NAMES))
else:
    print(f"⚠️  data.yaml non trouvé à {yaml_path}")
    CLASS_NAMES = ["tumour"]
    NC = 1


In [ ]:
# Cellule 2.4 — Comptage images/annotations par split
splits = ["train", "valid", "test"]
summary_rows = []

for split in splits:
    img_dir = PROCESSED_DETECTION_DIR / split / "images"
    lbl_dir = PROCESSED_DETECTION_DIR / split / "labels"
    
    images = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.bmp"))) if img_dir.exists() else []
    labels = sorted(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else []
    
    img_stems = {f.stem for f in images}
    lbl_stems = {f.stem for f in labels}
    matched = img_stems & lbl_stems
    orphan_img = img_stems - lbl_stems
    orphan_lbl = lbl_stems - img_stems
    
    # Distribution des classes
    class_counts = {i: 0 for i in range(NC)}
    for lbl_stem in matched:
        lbl_path = lbl_dir / f"{lbl_stem}.txt"
        try:
            with open(lbl_path) as f:
                for line in f:
                    line = line.strip()
                    if line:
                        cid = int(line.split()[0])
                        if 0 <= cid < NC:
                            class_counts[cid] += 1
        except Exception:
            pass
    
    summary_rows.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "matched": len(matched),
        "orphan_images": len(orphan_img),
        "orphan_labels": len(orphan_lbl),
        **{f"class_{i}": class_counts[i] for i in range(NC)},
    })
    
    print(f"{split:6s} : {len(images):5d} images | {len(labels):5d} labels | {len(matched):5d} paires | orphelins: {len(orphan_img)} img, {len(orphan_lbl)} lbl")
    for i in range(NC):
        name = CLASS_NAMES[i] if i < len(CLASS_NAMES) else f"class_{i}"
        print(f"         class {i} ({name}): {class_counts[i]} instances")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORTS_DIR / "detection" / "split_summary.csv", index=False)
print(f"\nRésumé sauvegardé : {REPORTS_DIR / 'detection' / 'split_summary.csv'}")


### 2.5 — Visualisation d'échantillons avec bounding boxes

Affichage d'images avec leurs boîtes englobantes ground truth.

In [ ]:
# Cellule 2.5 — Fonction de visualisation YOLO
from PIL import Image
import random

def draw_yolo_boxes(image_path: Path, label_path: Path, class_names: list, max_boxes: int = 20):
    """Dessine les boîtes YOLO sur une image."""
    with Image.open(image_path) as im:
        img = np.array(im.convert("RGB"))
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(img)
    
    if label_path.exists():
        with open(label_path) as f:
            for i, line in enumerate(f):
                if i >= max_boxes:
                    break
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    continue
                cid, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                # Conversion normalisé -> pixels
                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                box_w = bw * w
                box_h = bh * h
                
                rect = patches.Rectangle(
                    (x1, y1), box_w, box_h,
                    linewidth=2, edgecolor='lime', facecolor='none'
                )
                ax.add_patch(rect)
                label = class_names[cid] if cid < len(class_names) else f"class_{cid}"
                ax.text(x1, y1 - 5, label, color='lime', fontsize=12,
                       bbox=dict(facecolor='black', alpha=0.7, pad=1, edgecolor='none'))
    
    ax.set_title(f"{image_path.name}", fontsize=14, weight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# Affichage d'un échantillon par split
for split in splits:
    img_dir = PROCESSED_DETECTION_DIR / split / "images"
    lbl_dir = PROCESSED_DETECTION_DIR / split / "labels"
    
    if not img_dir.exists():
        continue
    
    images = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
    if not images:
        continue
    
    print(f"\n=== {split.upper()} - Échantillons ===")
    sample = random.sample(images, min(4, len(images)))
    for img_path in sample:
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        draw_yolo_boxes(img_path, lbl_path, CLASS_NAMES)


### 2.6 — Validation du format YOLO

Vérification détaillée de la validité des annotations.

In [ ]:
# Cellule 2.6 — Validation complète du format
from brainscan.training.validate import validate_detection

val_results = validate_detection(PROCESSED_DETECTION_DIR, CLASS_NAMES)

print(f"\nTotal images: {val_results['total_images']}")
print(f"Total labels: {val_results['total_labels']}")
print(f"Classes: {val_results['classes']}")
print(f"Class IDs trouvés: {val_results['class_ids_found']}")

if val_results['errors']:
    print(f"\n❌ {len(val_results['errors'])} ERREURS :")
    for e in val_results['errors'][:20]:
        print(f"  - {e}")
else:
    print("\n✅ Aucune erreur de format")

if val_results['warnings']:
    print(f"\n⚠️  {len(val_results['warnings'])} AVERTISSEMENTS :")
    for w in val_results['warnings']:
        print(f"  - {w}")

# Sauvegarde rapport
report_dir = REPORTS_DIR / "detection"
report_dir.mkdir(parents=True, exist_ok=True)
(report_dir / "validation_report.json").write_text(
    json.dumps(val_results, indent=2, default=str)
)

### 2.7 — Configuration de l'augmentation de données YOLOv8

YOLOv8 intègre nativement de nombreuses augmentations. Voici les paramètres recommandés pour l'imagerie médicale :

In [ ]:
# Cellule 2.7 — Paramètres d'augmentation YOLOv8
# Voir https://docs.ultralytics.com/usage/cfg-augmentation/

augmentation_config = {
    # Géométriques
    "degrees": 10.0,          # Rotation ±10°
    "translate": 0.1,         # Translation ±10%
    "scale": 0.5,             # Zoom 0.5-1.5x
    "shear": 2.0,             # Cisaillement ±2°
    "perspective": 0.0,       # Perspective (désactivé pour médical)
    "flipud": 0.0,            # Flip vertical (rarement pertinent IRM)
    "fliplr": 0.5,            # Flip horizontal 50%
    "mosaic": 1.0,            # Mosaic augmentation (4 images)
    "mixup": 0.1,             # MixUp 10%
    "copy_paste": 0.0,        # Copy-paste (désactivé)
    
    # Photométriques
    "hsv_h": 0.015,           # Teinte
    "hsv_s": 0.7,             # Saturation
    "hsv_v": 0.4,             # Valeur (luminosité)
    
    # Avancées
    "auto_augment": "randaugment",
    "erasing": 0.4,           # Random erasing
}

print("Configuration d'augmentation recommandée :")
for k, v in augmentation_config.items():
    print(f"  {k}: {v}")

# Sauvegarde pour référence
(REPORTS_DIR / "detection" / "augmentation_config.json").write_text(
    json.dumps(augmentation_config, indent=2)
)

### 2.8 — Entraînement YOLOv8

Lancement de l'entraînement avec les hyperparamètres choisis.

In [ ]:
# Cellule 2.8 — Lancement entraînement via script CLI
import subprocess
import time

if not yaml_path.exists():
    print("❌ data.yaml manquant — impossible de lancer l'entraînement")
else:
    cmd = [
        sys.executable, "-m", "brainscan.training.detection",
        "--data-yaml", str(yaml_path),
        "--models-dir", str(MODELS_DIR / "detection"),
        "--artifacts-dir", str(REPORTS_DIR / "detection"),
        "--epochs", "100",
        "--batch-size", "16",
        "--img-size", "640",
        "--model-size", "n",  # n, s, m, l, x
        "--lr0", "0.01",
        "--lrf", "0.01",
        "--patience", "20",
        "--device", "",  # auto
        "--workers", "8",
        "--seed", "42",
        "--mlflow-uri", os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"),
        "--mlflow-experiment", os.environ.get("MLFLOW_EXPERIMENT_NAME", "brainscan-v2"),
    ]
    
    print(f"Commande : {' '.join(cmd)}")
    print("\n⏳ Lancement de l'entraînement... (cela peut prendre du temps)")
    
    # Décommenter pour lancer réellement :
    # result = subprocess.run(cmd, capture_output=True, text=True)
    # print(result.stdout)
    # if result.stderr:
    #     print("STDERR:", result.stderr)
    print("\n⚠️  Exécution commentée — décommentez les lignes ci-dessus pour lancer.")


### 2.9 — Évaluation et analyse des résultats

Après entraînement, analyse des métriques (mAP@0.5, mAP@0.5:0.95, precision, recall),
visualisation des courbes, matrice de confusion, et vérification de la généralisation.

In [ ]:
# Cellule 2.9 — Analyse post-entraînement (à exécuter après entraînement)
from ultralytics import YOLO

model_path = MODELS_DIR / "detection" / "yolov8n_best.pt"

if model_path.exists():
    print(f"Chargement du modèle : {model_path}")
    model = YOLO(str(model_path))
    
    # Évaluation sur test
    results = model.val(
        data=str(yaml_path),
        split="test",
        imgsz=640,
        batch=16,
        verbose=True,
    )
    
    # Affichage métriques clés
    if hasattr(results, "results_dict"):
        rd = results.results_dict
        print(f"\n📊 Métriques TEST :")
        print(f"  mAP@0.5       : {rd.get('metrics/mAP50(B)', 0):.4f}")
        print(f"  mAP@0.5:0.95  : {rd.get('metrics/mAP50-95(B)', 0):.4f}")
        print(f"  Precision     : {rd.get('metrics/precision(B)', 0):.4f}")
        print(f"  Recall        : {rd.get('metrics/recall(B)', 0):.4f}")
        print(f"  F1-score      : {2*rd.get('metrics/precision(B)',0)*rd.get('metrics/recall(B)',0)/(rd.get('metrics/precision(B)',0)+rd.get('metrics/recall(B)',0)+1e-9):.4f}")
    
    # Matrice de confusion
    if hasattr(results, "confusion_matrix") and results.confusion_matrix is not None:
        cm = results.confusion_matrix.matrix
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                   xticklabels=CLASS_NAMES + ["background"],
                   yticklabels=CLASS_NAMES + ["background"],
                   ax=ax)
        ax.set_title("Matrice de confusion - TEST", weight="bold")
        ax.set_xlabel("Prédit")
        ax.set_ylabel("Réel")
        plt.tight_layout()
        plt.show()
    
    # Courbes d'entraînement (depuis results.csv)
    csv_path = MODELS_DIR / "detection" / "train" / "results.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes[0,0].plot(df['epoch'], df['train/box_loss'], label='box_loss')
        axes[0,0].plot(df['epoch'], df['train/cls_loss'], label='cls_loss')
        axes[0,0].plot(df['epoch'], df['train/dfl_loss'], label='dfl_loss')
        axes[0,0].set_title('Losses')
        axes[0,0].legend()
        axes[0,0].grid(alpha=0.3)
        
        axes[0,1].plot(df['epoch'], df['metrics/precision(B)'], label='precision')
        axes[0,1].plot(df['epoch'], df['metrics/recall(B)'], label='recall')
        axes[0,1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50')
        axes[0,1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95')
        axes[0,1].set_title('Métriques validation')
        axes[0,1].legend()
        axes[0,1].grid(alpha=0.3)
        
        axes[1,0].plot(df['epoch'], df['lr/pg0'], label='lr_pg0')
        axes[1,0].plot(df['epoch'], df['lr/pg1'], label='lr_pg1')
        axes[1,0].plot(df['epoch'], df['lr/pg2'], label='lr_pg2')
        axes[1,0].set_title('Learning rates')
        axes[1,0].legend()
        axes[1,0].grid(alpha=0.3)
        
        axes[1,1].plot(df['epoch'], df['val/box_loss'], label='val_box')
        axes[1,1].plot(df['epoch'], df['val/cls_loss'], label='val_cls')
        axes[1,1].plot(df['epoch'], df['val/dfl_loss'], label='val_dfl')
        axes[1,1].set_title('Validation losses')
        axes[1,1].legend()
        axes[1,1].grid(alpha=0.3)
        
        plt.suptitle("Courbes d'entraînement YOLOv8", fontsize=14, weight="bold")
        plt.tight_layout()
        plt.show()
else:
    print(f"Modèle non trouvé : {model_path}")
    print("Lancez d'abord l'entraînement (cellule précédente).")


### 2.10 — Inférence et visualisation des prédictions

Test du modèle sur des images de test avec affichage des prédictions.

In [ ]:
# Cellule 2.10 — Inférence sur échantillon test
import random

if model_path.exists():
    model = YOLO(str(model_path))
    
    test_img_dir = PROCESSED_DETECTION_DIR / "test" / "images"
    test_lbl_dir = PROCESSED_DETECTION_DIR / "test" / "labels"
    
    if test_img_dir.exists():
        test_images = sorted(list(test_img_dir.glob("*.jpg")) + list(test_img_dir.glob("*.png")))
        sample = random.sample(test_images, min(6, len(test_images)))
        
        for img_path in sample:
            # Prédiction
            results = model.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)
            r = results[0]
            
            # Ground truth
            lbl_path = test_lbl_dir / f"{img_path.stem}.txt"
            
            # Affichage côte à côte
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
            
            # Ground truth
            with Image.open(img_path) as im:
                img_arr = np.array(im.convert("RGB"))
            h, w = img_arr.shape[:2]
            ax1.imshow(img_arr)
            if lbl_path.exists():
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            cid, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                            x1 = (xc - bw/2) * w
                            y1 = (yc - bh/2) * h
                            rect = patches.Rectangle((x1, y1), bw*w, bh*h, linewidth=2, edgecolor='lime', facecolor='none')
                            ax1.add_patch(rect)
                            ax1.text(x1, y1-5, CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else f"c{cid}",
                                    color='lime', fontsize=10, bbox=dict(facecolor='black', alpha=0.7, pad=1))
            ax1.set_title("Ground Truth", weight="bold")
            ax1.axis('off')
            
            # Prédiction
            ax2.imshow(img_arr)
            if r.boxes is not None:
                for box in r.boxes:
                    xyxy = box.xyxy[0].cpu().numpy()
                    conf = float(box.conf[0])
                    cid = int(box.cls[0])
                    x1, y1, x2, y2 = xyxy
                    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
                    ax2.add_patch(rect)
                    label = f"{CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else f'c{cid}'} {conf:.2f}"
                    ax2.text(x1, y1-5, label, color='red', fontsize=10,
                            bbox=dict(facecolor='black', alpha=0.7, pad=1))
            ax2.set_title("Prédiction YOLOv8", weight="bold")
            ax2.axis('off')
            
            plt.suptitle(f"Test : {img_path.name}", fontsize=14, weight="bold")
            plt.tight_layout()
            plt.show()
else:
    print("Modèle non entraîné — lancez l'entraînement d'abord.")


### 2.11 — Export et sauvegarde finale

Sauvegarde du modèle au format ONNX pour déploiement, et résumé final.

In [ ]:
# Cellule 2.11 — Export ONNX et résumé
if model_path.exists():
    model = YOLO(str(model_path))
    
    # Export ONNX
    onnx_path = MODELS_DIR / "detection" / "yolov8n_best.onnx"
    try:
        exported = model.export(format="onnx", imgsz=640, simplify=True, opset=12)
        print(f"✅ Export ONNX : {exported}")
        if Path(exported).exists():
            import shutil
            shutil.move(exported, onnx_path)
            print(f"   Déplacé vers : {onnx_path}")
    except Exception as e:
        print(f"❌ Erreur export ONNX : {e}") #onnx means open neural network exchange, a format for representing deep learning models
    
    # Métadonnées finales
    metadata = {
        "task": "brain_tumor_detection",
        "architecture": "YOLOv8n",
        "framework": "ultralytics",
        "input_size": [640, 640],
        "classes": CLASS_NAMES,
        "model_path": str(model_path),
        "onnx_path": str(onnx_path) if onnx_path.exists() else None,
        "data_yaml": str(yaml_path),
        "trained_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    (MODELS_DIR / "detection" / "metadata.json").write_text(json.dumps(metadata, indent=2))
    print(f"Métadonnées sauvegardées : {MODELS_DIR / 'detection' / 'metadata.json'}")
else:
    print("Modèle non disponible pour l'export.")


### 2.12 — Récapitulatif de l'étape 2

L'étape 2 (détection YOLOv8) comprend :

* **Données** : Format YOLO (images + labels .txt), split 70/15/15, data.yaml créé
* **Validation** : Vérification format, cohérence classes, paires image/label
* **Augmentation** : Mosaic, MixUp, HSV, flip, rotation, translation, scale
* **Entraînement** : YOLOv8n (ou s/m/l/x), 100 epochs, batch 16, imgsz 640
* **Métriques** : mAP@0.5, mAP@0.5:0.95, Precision, Recall, F1
* **Analyse** : Courbes d'apprentissage, matrice de confusion, prédictions vs GT
* **Export** : Modèle .pt (PyTorch) + .onnx (déploiement), métadonnées JSON
* **MLflow** : Hyperparamètres, métriques, artefacts, modèle loggés

**Suite** : Étape 3 (MLflow avancé), Étape 4 (Airflow DAG), Étape 5 (API + Streamlit).